## **RAG with Vector Search**

Previous lessons code to generate the objects for this lesson

In [10]:
# import the documents
from ingest import load_faq_data

documents = load_faq_data()

# embed the question and answer so the query can match against the question text and answer text in our index 
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

# import the model
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

# encode the embeddings in batch of 50
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

# turn them into 2 dimensional array(matrix) where rows are documents(vectors) and columns are dimensions of the vectors
import numpy as np
X = np.array(vectors)

# vector index the documents with minsearch
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  0%|          | 0/27 [00:00<?, ?it/s]

In module 1, we built a RAG pipeline with three steps:

In [ ]:
# def rag(question):
#     search_results = search(question)
#     user_prompt = build_prompt(question, search_results)
#     return llm(user_prompt)

The search step used keyword search. Now we swap in vector search. Because RAG is modular, search is the only step we touch. Build prompt and the LLM call stay exactly as they were.

## Using RAGBase

In module 1 we put all the RAG logic into a RAGBase helper class. It has search, build_prompt, and llm methods, so we only need to override search.

Download rag_helper.py (and ingest.py if you didn't get it earlier) into your project:

In [1]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
# wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-27 11:57:20--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0.001s  

2026-06-27 11:57:20 (3.50 MB/s) - ‘rag_helper.py’ saved [2134/2134]



First, create the OpenAI client:

In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

Next, download and index the data:

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

Then use the RAGBase class:

In [ ]:
# from rag_helper import RAGBase

# assistant = RAGBase(
#     index=index,
#     llm_client=openai_client,
# )

# query = "I just found out about the program, can I still sign up?"
# assistant.rag(query)

This still uses keyword search. Text search isn't bad here, so the answer may already look right. Next we replace search with vector search.

We already have:

All the indexed documents documents
The embeddings matrix X with all these documents
The vector search engine vindex
We can't pass vindex to RAG as-is. Text search takes the query string directly, but vector search needs the query as a vector first. So we subclass RAGBase and override search to encode the query before searching.

The subclass overrides search:

In [ ]:
# class RAGVector(RAGBase):

#     def __init__(self, embedder, **kwargs):
#         super().__init__(**kwargs)
#         self.embedder = embedder

#     def search(self, query, num_results=5):
#         query_vector = self.embedder.encode(query)
#         filter_dict = {"course": self.course}

#         return self.index.search(
#             query_vector,
#             num_results=num_results,
#             filter_dict=filter_dict
#         )

The __init__ method adds one extra argument, embedder, for the sentence transformer. Inside search we use it to turn the query into a vector. Then we query vindex with that vector instead of the raw text. Everything else is inherited from RAGBase.

## Using it

Let's init it:

In [11]:
from rag_helper import RAGVector

vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes — you can still join even if the program has already begun.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.'

The answers should be close to what we got with keyword search, but vector search handles rephrased questions better. The swap was trivial because RAG has three clear steps. The same trick lets us change the LLM provider later by overriding just the llm step.